# Set 15 – Modelle speichern und laden: scikit-learn
Wir trainieren einmal eine kleine Iris-Klassifikation. Anschließend speichern wir **Scaler und Modell gemeinsam**, laden sie neu und prüfen die Vorhersagen. Der neue Prozess kennt nur Modelldateien und neue Eingaben; er trainiert nichts nach.

Ablauf: Training → Pipeline + Konfiguration speichern → laden → Eingaben prüfen → vorhersagen.

In [1]:
from pathlib import Path
import sys
kandidaten=[Path.cwd(),*Path.cwd().parents]
set_ordner=next(p for basis in kandidaten for p in (basis,basis/'set15-modelle-speichern-und-laden') if (p/'deployment_tools.py').exists())
if str(set_ordner) not in sys.path:sys.path.insert(0,str(set_ordner))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from deployment_tools import ARTEFAKTE,umgebung,json_speichern,eingaben_pruefen,neuer_prozess

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib
iris=load_iris(as_frame=True)
X_train,X_test,y_train,y_test=train_test_split(iris.data,iris.target,test_size=0.25,random_state=42,stratify=iris.target)

## 1. Preprocessing und Modell zusammen trainieren
Der Scaler lernt ausschließlich aus Trainingsdaten. Die Feature-Namen und die Klassenbezeichnungen gehören ebenfalls zum Modellpaket.

In [3]:
pipeline=Pipeline([('scaler',StandardScaler()),('modell',LogisticRegression(max_iter=300))])
pipeline.fit(X_train,y_train)
print('Test-Accuracy:',pipeline.score(X_test,y_test))
display(X_test.head())

Test-Accuracy: 0.9210526315789473


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
42,4.4,3.2,1.3,0.2
56,6.3,3.3,4.7,1.6
99,5.7,2.8,4.1,1.3
53,5.5,2.3,4.0,1.3
38,4.4,3.0,1.3,0.2


## 2. Speichern
`joblib.dump` speichert die komplette gefittete Pipeline. JSON enthält lesbare Begleitinformationen; die Versionsdatei dokumentiert die verwendete Umgebung.

Joblib/Pickle-Dateien können beim Laden Code ausführen. Wir laden hier nur unsere selbst erzeugten Dateien. Das Laden mit anderen scikit-learn-Versionen wird nicht unterstützt; die Trainingsumgebung sollte reproduziert werden.

In [4]:
ordner=ARTEFAKTE/'sklearn_pipeline';ordner.mkdir(exist_ok=True)
config={'feature_namen':list(X_train.columns),'klassen':iris.target_names.tolist(),'modell':'StandardScaler + LogisticRegression'}
joblib.dump(pipeline,ordner/'pipeline.joblib')
json_speichern(ordner/'config.json',config)
json_speichern(ordner/'versionen.json',umgebung(['scikit-learn','numpy','pandas','joblib']))
display(pd.DataFrame({'Datei':[p.name for p in ordner.iterdir()],'Bytes':[p.stat().st_size for p in ordner.iterdir()]}))

,Datei,Bytes
0,config.json,259
1,pipeline.joblib,1873
2,versionen.json,121


## 3. Laden und Ergebnisvergleich
Ein neues Python-Objekt reicht für den ersten Test. Der zweite Test startet einen separaten Python-Prozess über `predict.py`, der die Pipeline ohne `fit()` lädt.

In [5]:
neu=joblib.load(ordner/'pipeline.joblib')
np.testing.assert_allclose(neu.predict_proba(X_test),pipeline.predict_proba(X_test),rtol=0,atol=0)
probe=X_test.iloc[:6]
extern=neuer_prozess('sklearn',ordner,probe.to_dict(orient='records'))
np.testing.assert_allclose(extern['wahrscheinlichkeiten'],pipeline.predict_proba(probe),rtol=1e-12,atol=1e-12)
assert extern['labels']==pipeline.predict(probe).tolist()
print('Roundtrip und neuer Prozess: gleiche Klassen und Wahrscheinlichkeiten.')
display(pd.DataFrame(extern['wahrscheinlichkeiten'],columns=iris.target_names))

Neuer Python-Prozess: Modell geladen; Vorhersagen ohne Training: 6
Roundtrip und neuer Prozess: gleiche Klassen und Wahrscheinlichkeiten.


,setosa,versicolor,virginica
0,0.984579,0.015421,3.686584e-07
1,0.033614,0.621377,3.450094e-01
2,0.045086,0.882484,7.243009e-02
3,0.010807,0.916294,7.289940e-02
4,0.967512,0.032487,8.096882e-07
5,0.101060,0.634043,2.648969e-01


## 4. Feature-Struktur kontrollieren
Andere Spaltenreihenfolge wird anhand der gespeicherten Namen korrigiert. Fehlende/zusätzliche Features oder ungültige Werte werden abgewiesen. Diese Prüfung sitzt vor der gespeicherten Pipeline.

In [6]:
anders=probe.loc[:,list(reversed(config['feature_namen']))]
geordnet=eingaben_pruefen(anders,config['feature_namen'])
np.testing.assert_allclose(neu.predict_proba(geordnet),neu.predict_proba(probe))
try:
    eingaben_pruefen(probe.drop(columns=config['feature_namen'][0]),config['feature_namen'])
except ValueError as exc:print('Erwarteter Eingabefehler:',exc)
else:raise AssertionError('Fehlende Spalte hätte abgewiesen werden müssen.')

Erwarteter Eingabefehler: Feature-Namen passen nicht: fehlende oder zusätzliche Spalten.


**Merken:** Die Pipeline, Feature-Struktur und passende Umgebung gehören zusammen. Zum Vorhersagen werden die gespeicherten Transformationen verwendet; der Scaler wird nicht erneut gefittet.

Quelle: [scikit-learn: Model persistence](https://scikit-learn.org/stable/model_persistence.html). PDF-Bezug: Seiten 3–6, 8 und 12–15.